# Notebook 01: Carga y traducción de los datos
**Proyecto:** Predicción de Insatisfacción del Cliente — Olist (Grupo C)

**Objetivo:** cargar los 7 CSV originales, traducir al español lo que haga falta y guardar las
tablas listas para que el Notebook 02 empiece a limpiar y preparar los datos.

**Pasos:**
1. Carga de los 7 datasets originales (`data/`).
2. Traducción al español (categorías, estado del pedido, tipo de pago, región).
3. Guardado de las tablas traducidas en `data/procesado/`.

**Entrada:** CSV originales de Olist. **Salida:** 7 archivos `*_traducida.csv` en `data/procesado/`.

In [ ]:
import os
import sys
import pandas as pd

# Las funciones propias están en la carpeta src (una carpeta arriba del notebook)
sys.path.append(os.path.abspath('../SRC'))
from preparacion import cargar_crudos, traducir, guardar_traducidas

pd.set_option('display.max_columns', 60)

RUTA_DATOS = '../data'
RUTA_PROC = '../data/procesado'

## 1. Carga
Se cargan los 7 datasets originales. La función `cargar_crudos` (en `src/preparacion.py`) convierte
las fechas y lee los códigos postales como texto para no perder los ceros iniciales.

In [8]:
dfs = cargar_crudos(RUTA_DATOS)

for nombre, df in dfs.items():
    print(f'{nombre:10}: {df.shape[0]:>7} filas, {df.shape[1]:>2} columnas')

orders    :   99441 filas,  8 columnas
items     :  112650 filas,  7 columnas
payments  :  103886 filas,  5 columnas
reviews   :   99224 filas,  7 columnas
customers :   99441 filas,  5 columnas
products  :   32951 filas,  9 columnas
sellers   :    3095 filas,  4 columnas


## 2. Traducción

### ¿Qué se traduce?
| Variable | Idioma original | Decisión |
|---|---|---|
| Nombres de columnas | Inglés | Se mantienen |
| `product_category_name` | Portugués | Se traduce al español (73 categorías, diccionario propio) |
| `order_status`, `payment_type` | Inglés | Se traducen al español (uniformidad en el informe) |
| `customer_state`, `seller_state` | Sigla | Se agrega la región de Brasil |
| `customer_city`, `seller_city` | Portugués | No se traducen (nombres propios) |
| `review_comment_message` | Portugués | No se traduce: se escribe junto con la puntuación (fuga de información) |

Los diccionarios están en `src/diccionarios.py`. El archivo de traducción del dataset
(al inglés) se descartó porque le faltaban 2 categorías y tenía errores de tipeo.

In [9]:
dfs = traducir(dfs)

# Controles: ningún valor debe quedar sin traducir
p = dfs['products']
print('Categorías sin traducir:', ((p['categoria_producto'] == 'sin_categoria')
                                   & p['product_category_name'].notna()).sum())
print('Productos sin categoría (nulos de origen):', p['product_category_name'].isna().sum())
print('Estados de pedido sin traducir:', dfs['orders']['estado_pedido'].isna().sum())
print('Tipos de pago sin traducir:', dfs['payments']['tipo_pago'].isna().sum())
print('Clientes sin región:', dfs['customers']['customer_region'].isna().sum())
print('Vendedores sin región:', dfs['sellers']['seller_region'].isna().sum())

Categorías sin traducir: 0
Productos sin categoría (nulos de origen): 610
Estados de pedido sin traducir: 0
Tipos de pago sin traducir: 0
Clientes sin región: 0
Vendedores sin región: 0


## 3. Guardado de las tablas traducidas
Se guarda cada tabla en `data/procesado/` . El Notebook 02 las lee
con `cargar_traducidas`, que vuelve a convertir las fechas y los códigos postales.

In [10]:
guardar_traducidas(dfs, RUTA_PROC)

for nombre in dfs:
    archivo = os.path.join(RUTA_PROC, f'{nombre}_traducida.csv')
    print(f'{nombre:10}: {archivo}  ({os.path.getsize(archivo) / 1e6:.1f} MB)')

orders    : ../data/procesado\orders_traducida.csv  (17.6 MB)
items     : ../data/procesado\items_traducida.csv  (15.0 MB)
payments  : ../data/procesado\payments_traducida.csv  (7.4 MB)
reviews   : ../data/procesado\reviews_traducida.csv  (14.1 MB)
customers : ../data/procesado\customers_traducida.csv  (9.5 MB)
products  : ../data/procesado\products_traducida.csv  (3.4 MB)
sellers   : ../data/procesado\sellers_traducida.csv  (0.2 MB)


## Resumen
- Se cargaron los **7 datasets** de Olist y se tradujeron al español la categoría del producto, el estado
  del pedido, el tipo de pago y la región de clientes y vendedores (los nombres de columnas se mantienen).
- Los controles no encontraron valores sin traducir (salvo los productos sin categoría de origen).
- **Siguiente paso:** Notebook 02 (limpieza y preparación de la base consolidada).